In [1]:
# 1. Install and Import
!pip install tcia_utils -q
from tcia_utils import nbia
import os

# 2. Get the list of series for the collection
# We fetch metadata first because downloadSeries requires specific UIDs
series_data = nbia.getSeries(collection = "NSCLC-Radiomics")

# 3. Select the first 5 series for testing
# This prevents downloading the entire massive dataset at once
test_series = series_data[:5]

# 4. Download the selected series
# The function will now receive the correct list format
nbia.downloadSeries(test_series, path = "/content/Dataset/LDCT_Invasive")

print("\n✅ Success! 5 LDCT series from NSCLC-Radiomics are now in Colab.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.3 MB/s eta 0:00:00

✅ Success! 5 LDCT series from NSCLC-Radiomics are now in Colab.


In [5]:
# Ambil metadata dari koleksi LDCT (Low Dose CT)
normal_series_data = nbia.getSeries(collection = "LDCT-and-Projection-data")

# Kita ambil 5 series saja agar seimbang dengan data Nodule tadi
test_normal_series = normal_series_data[:5]

# Download ke folder terpisah agar mudah diberi label 0
nbia.downloadSeries(test_normal_series, path = "/content/Dataset/LDCT_Normal")

print("\n✅ Data Paru-Paru 'Normal' berhasil ditarik!")


✅ Data Paru-Paru 'Normal' berhasil ditarik!


In [2]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.6 MB/s eta 0:00:00


In [3]:
import os
import pydicom
import numpy as np
import cv2
from tqdm import tqdm
import glob

def convert_dicom_to_png(src_dir, out_dir):
    """
    Finds all .dcm files in subdirectories and converts them to PNG.
    src_dir: The 'LDCT_Invasive' folder containing nested DICOM files.
    out_dir: The new folder to store PNG images.
    """
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # Find all .dcm files recursively
    dicom_files = glob.glob(os.path.join(src_dir, "**", "*.dcm"), recursive=True)

    print(f"📦 Found {len(dicom_files)} DICOM files. Starting conversion...")

    for dcm_path in tqdm(dicom_files, desc="Converting"):
        try:
            # Read DICOM file
            ds = pydicom.dcmread(dcm_path)
            pixel_array = ds.pixel_array

            # Basic Normalization (Scaling 16-bit to 8-bit for PNG)
            # Medical images often have high dynamic range
            img = pixel_array.astype(np.float32)
            img = (img - np.min(img)) / (np.max(img) - np.min(img)) * 255.0
            img = img.astype(np.uint8)

            # Create a unique name based on PatientID and InstanceNumber
            patient_id = ds.PatientID if 'PatientID' in ds else "Unknown"
            instance_num = ds.InstanceNumber if 'InstanceNumber' in ds else os.path.basename(dcm_path)
            file_name = f"{patient_id}_slice_{instance_num}.png"

            # Save as PNG
            cv2.imwrite(os.path.join(out_dir, file_name), img)

        except Exception as e:
            # Skip corrupted files or files without pixel data
            continue

    print(f"✅ Conversion complete! Files saved in: {out_dir}")

# --- EXECUTION ---
DICOM_DIR = "/content/Dataset/LDCT_Invasive"
PNG_DIR = "/content/Dataset/LDCT_PNG" # We will use this path for training

convert_dicom_to_png(DICOM_DIR, PNG_DIR)

📦 Found 757 DICOM files. Starting conversion...


Converting: 100%|██████████| 757/757 [00:04<00:00, 176.97it/s]

✅ Conversion complete! Files saved in: /content/Dataset/LDCT_PNG


In [6]:
convert_dicom_to_png("/content/Dataset/LDCT_Normal", "/content/Dataset/LDCT_Normal_PNG")

📦 Found 62948 DICOM files. Starting conversion...


Converting: 100%|██████████| 62948/62948 [03:38<00:00, 287.98it/s]

✅ Conversion complete! Files saved in: /content/Dataset/LDCT_Normal_PNG


In [7]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd
import pywt
import scipy.stats
import warnings
import glob
import gc
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.utils import to_categorical, Sequence

# --- CONFIGURATIONS & CUDA SETUP ---
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Setting up GPU Memory Growth to prevent sudden spikes
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"✅ GPU Ready: {physical_devices[0].name}")
else:
    print("⚠️ Running on CPU mode.")

IMG_SIZE = (256, 256)
HC_IMG_SIZE = (128, 128) # Smaller size for faster texture extraction
WAVELET = 'db1'
CLASS_NAMES = ['Normal', 'Nodule'] # Label 0: Normal, Label 1: Nodule
NUM_CLASSES = len(CLASS_NAMES)
SAVE_DIR = "./lidc_nodule_model"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# ==============================================================================
# 1. FEATURE EXTRACTION (HANDCRAFTED)
# ==============================================================================
def extract_handcrafted(img):
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img

    # Wavelet Transform (Discrete)
    coeffs = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs

    def _stats(band):
        flat = np.abs(band.flatten()) + 1e-6
        return [np.mean(band), np.std(band), np.var(band), scipy.stats.entropy(flat)]

    feats = []
    for band in [LL, LH, HL, HH]: feats.extend(_stats(band))

    # GLCM Features
    from skimage.feature import graycomatrix, graycoprops
    gray_norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    glcm = graycomatrix(gray_norm, [5], [0, np.pi/4, np.pi/2], 256, symmetric=True, normed=True)
    feats.extend([graycoprops(glcm, 'contrast').mean(),
                  graycoprops(glcm, 'dissimilarity').mean(),
                  graycoprops(glcm, 'homogeneity').mean()])
    return feats

# ==============================================================================
# 2. DATA GENERATOR (RAM EFFICIENT) - SYNCED VERSION
# ==============================================================================
class LIDCDataGenerator(Sequence):
    def __init__(self, file_paths, hc_features, labels, batch_size=16, img_size=(256, 256)):
        self.file_paths = file_paths
        self.hc_features = hc_features
        self.labels = labels
        self.batch_size = batch_size
        self.img_size = img_size

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / self.batch_size))

    def __getitem__(self, idx):
        # Slice data for the current batch
        batch_paths = self.file_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_hc = self.hc_features[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_y = self.labels[idx * self.batch_size : (idx + 1) * self.batch_size]

        images = []
        for path in batch_paths:
            img = cv2.imread(path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, self.img_size)
                # Cast to float32 to save RAM during training
                images.append(img.astype(np.float32) / 255.0)
            else:
                # Handle missing images with a zero-filled placeholder
                images.append(np.zeros((self.img_size[0], self.img_size[1], 3), dtype=np.float32))

        # Dictionary mapping to layer names to avoid TypeErrors
        inputs = {
            "img_input": np.array(images),
            "hc_input": np.array(batch_hc)
        }

        return inputs, np.array(batch_y)

# ==============================================================================
# 3. DATA LOADING (LDCT INVASIVE STRUCTURE)
# ==============================================================================
def load_dataset_efficient(normal_dir, invasive_dir):
    file_paths, hc_list, labels = [], [], []

    # 1. Load Data Normal (Label 0)
    normal_files = glob.glob(os.path.join(normal_dir, "*.png"))
    for p in normal_files:
        img = cv2.imread(p)
        if img is not None:
            hc_feats = extract_handcrafted(cv2.resize(img, HC_IMG_SIZE))
            file_paths.append(p)
            hc_list.append(hc_feats)
            labels.append(0) # Label Normal

    # 2. Load Data Invasive (Label 1)
    invasive_files = glob.glob(os.path.join(invasive_dir, "*.png"))
    for p in invasive_files:
        img = cv2.imread(p)
        if img is not None:
            hc_feats = extract_handcrafted(cv2.resize(img, HC_IMG_SIZE))
            file_paths.append(p)
            hc_list.append(hc_feats)
            labels.append(1) # Label Nodule/Invasive

    unique, counts = np.unique(labels, return_counts=True)
    print(f"✅ Dataset Balanced! Distribution: {dict(zip(unique, counts))}")
    return file_paths, np.array(hc_list), np.array(labels)


# ==============================================================================
# 4. HYBRID ARCHITECTURE & PIPELINE
# ==============================================================================
def build_hybrid_model(hc_dim):
    # CNN Branch
    img_input = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="img_input")
    x = Conv2D(32, (3, 3), activation='relu')(img_input)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    cnn_out = Dense(128, activation='relu')(x)

    # Handcrafted Branch
    hc_input = Input(shape=(hc_dim,), name="hc_input")
    hc_out = Dense(32, activation='relu')(hc_input)

    # Merge
    merged = Concatenate()([cnn_out, hc_out])
    output = Dense(NUM_CLASSES, activation='softmax')(merged)

    model = Model(inputs=[img_input, hc_input], outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def train_pipeline(data_dir):
    # STAGE 1: Data Acquisition
    NORM_PATH = "/content/Dataset/LDCT_Normal_PNG"
    INVA_PATH = "/content/Dataset/LDCT_PNG"

    # Update train_pipeline untuk menerima dua path ini jika perlu,
    # atau panggil load_dataset_efficient langsung di sini.
    paths, X_hc, y_labels = load_dataset_efficient(NORM_PATH, INVA_PATH)
    if not paths: return

    # Shuffle and Split
    paths, X_hc, y_labels = shuffle(paths, X_hc, y_labels, random_state=42)
    y_cat = to_categorical(y_labels, num_classes=NUM_CLASSES)

    scaler = StandardScaler()
    X_hc_scaled = scaler.fit_transform(X_hc)

    split = int(0.8 * len(paths))
    train_gen = LIDCDataGenerator(paths[:split], X_hc_scaled[:split], y_cat[:split])
    val_gen = LIDCDataGenerator(paths[split:], X_hc_scaled[split:], y_cat[split:])

    # STAGE 2: Keras Training (Uses Generator to save RAM)
    print("\n--- Training Hybrid CNN + Handcrafted ---")
    model = build_hybrid_model(X_hc.shape[1])
    model.fit(train_gen, validation_data=val_gen, epochs=10)

    # STAGE 3: Feature Extraction in Chunks
    print("\n--- Extracting Features for LightGBM Agent (Chunked) ---")
    extractor = Model(inputs=model.input, outputs=model.layers[-2].output)

    deep_feats = []
    CHUNK_SIZE = 50
    for i in range(0, len(paths), CHUNK_SIZE):
        c_paths = paths[i:i+CHUNK_SIZE]
        c_imgs = []
        for p in c_paths:
            img = cv2.imread(p)
            img = cv2.resize(img, (256,256))
            c_imgs.append(img/255.0)
        c_imgs = np.array(c_imgs).astype(np.float32)
        c_hc = X_hc_scaled[i:i+CHUNK_SIZE]

        preds = extractor.predict([c_imgs, c_hc], verbose=0)
        deep_feats.append(preds)

        # Explicitly clear chunk from memory
        del c_imgs
        gc.collect()

    X_final = np.vstack(deep_feats)

    # STAGE 4: LightGBM Training
    print("\n--- Training LightGBM Agent ---")
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_labels, test_size=0.2)

    lgb_data = lgb.Dataset(X_train, label=y_train)
    agent = lgb.train({'objective':'multiclass','num_class':NUM_CLASSES,'metric':'multi_logloss','verbose':-1}, lgb_data)

    # STAGE 5: Evaluation
    preds = np.argmax(agent.predict(X_test), axis=1)
    print(f"\n🎯 Final Agent Accuracy: {accuracy_score(y_test, preds)*100:.2f}%")
    print(classification_report(y_test, preds, target_names=CLASS_NAMES))

if __name__ == "__main__":
    DATA_PATH = "/content/Dataset/LDCT_PNG"
    train_pipeline(DATA_PATH)

✅ GPU Ready: /physical_device:GPU:0
✅ Dataset Balanced! Distribution: {np.int64(0): np.int64(37428), np.int64(1): np.int64(757)}

--- Training Hybrid CNN + Handcrafted ---
Epoch 1/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 95s 48ms/step - accuracy: 0.9978 - loss: 0.0452 - val_accuracy: 1.0000 - val_loss: 4.4487e-08
Epoch 2/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 86s 45ms/step - accuracy: 1.0000 - loss: 3.4651e-08 - val_accuracy: 1.0000 - val_loss: 9.8808e-09
Epoch 3/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 86s 45ms/step - accuracy: 1.0000 - loss: 9.5393e-09 - val_accuracy: 1.0000 - val_loss: 3.1375e-09
Epoch 4/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 85s 44ms/step - accuracy: 1.0000 - loss: 3.2455e-09 - val_accuracy: 1.0000 - val_loss: 1.1551e-09
Epoch 5/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 84s 44ms/step - accuracy: 1.0000 - loss: 1.1706e-09 - val_accuracy: 1.0000 - val_loss: 3.9024e-10
Epoch 6/10
1910/1910 ━━━━━━━━━━━━━━━━━━━━ 84s 44ms/step - accuracy: 1.0000 - loss: 4.3164e-10 - val_accuracy: 1.0000 - val_loss: 1.4